In [19]:
import pandas as pd
import numpy as np

tracking_df    = pd.read_csv('tracking.csv')
goal_prob_df   = pd.read_csv('goal_predictions.csv')
movement_df    = pd.read_csv('movement_features.csv')

print(f'Tracking rows:     {len(tracking_df)}')
print(f'Goal prob rows:    {len(goal_prob_df)}')
print(f'Movement rows:     {len(movement_df)}')
print(f'Total frames:      {tracking_df["frame"].nunique()}')

Tracking rows:     17398
Goal prob rows:    14715
Movement rows:     14938
Total frames:      750


2-Possession

In [20]:
# Possession = which team has a player closest to the ball each frame
# For each frame where ball is detected, find which team's player
# is nearest to the ball position

ball_df    = tracking_df[tracking_df['role'] == 'ball'][['frame','x','y']].rename(columns={'x':'ball_x','y':'ball_y'})
players_df = tracking_df[tracking_df['role'].isin(['player','goalkeeper'])].copy()

# Merge ball position into player data
merged = players_df.merge(ball_df, on='frame', how='inner')

# Calculate each player's distance to ball
merged['dist_to_ball'] = np.sqrt(
    (merged['x'] - merged['ball_x'])**2 +
    (merged['y'] - merged['ball_y'])**2
)

# For each frame find the player closest to ball
closest = merged.loc[merged.groupby('frame')['dist_to_ball'].idxmin()]

# Count frames each team was closest to ball
possession_counts = closest['team_id'].value_counts()
total             = possession_counts.sum()

possession = {
    0: round(possession_counts.get(0, 0) / total * 100, 1),
    1: round(possession_counts.get(1, 0) / total * 100, 1)
}

print('=== POSSESSION ===')
print(f'Team 0: {possession[0]}%')
print(f'Team 1: {possession[1]}%')

=== POSSESSION ===
Team 0: 83.4%
Team 1: 16.6%


3-Shots

In [21]:
# A shot = any frame where a player has goal probability above 0.5
# This is a simplified but reasonable definition

SHOT_THRESHOLD = 0.65

shots_df = goal_prob_df[goal_prob_df['goal_probability'] >= SHOT_THRESHOLD].copy()

# Count shots per team
shots = {
    0: len(shots_df[shots_df['team_id'] == 0]),
    1: len(shots_df[shots_df['team_id'] == 1])
}

# Also get average danger level per team
avg_danger = {
    0: round(goal_prob_df[goal_prob_df['team_id'] == 0]['goal_probability'].mean(), 3),
    1: round(goal_prob_df[goal_prob_df['team_id'] == 1]['goal_probability'].mean(), 3)
}

print('=== SHOTS (goal prob > 0.5) ===')
print(f'Team 0: {shots[0]} shots')
print(f'Team 1: {shots[1]} shots')
print()
print('=== AVERAGE DANGER LEVEL ===')
print(f'Team 0: {avg_danger[0]}')
print(f'Team 1: {avg_danger[1]}')

=== SHOTS (goal prob > 0.5) ===
Team 0: 7 shots
Team 1: 2 shots

=== AVERAGE DANGER LEVEL ===
Team 0: 0.262
Team 1: 0.266


4-Territory Control

In [22]:
# Territory = what fraction of time each team spends
# in the opponent's half of the pitch
# Pitch length = 105m, halfway = 52.5m

PITCH_X_MIN = 1405.4
PITCH_X_MAX = 11780.6
REAL_LENGTH  = 105

players_pitch = tracking_df[
    tracking_df['role'].isin(['player','goalkeeper'])
].dropna(subset=['pitch_x']).copy()

# Normalize pitch_x to meters
players_pitch['pitch_x_m'] = (
    (players_pitch['pitch_x'] - PITCH_X_MIN) /
    (PITCH_X_MAX - PITCH_X_MIN)
) * REAL_LENGTH

# Territory = fraction of frames spent in attacking half
# Team 0 attacks right half (>52.5m)
# Team 1 attacks left half (<52.5m)
# Note: if teams are reversed just swap, it still shows dominance

team0 = players_pitch[players_pitch['team_id'] == 0]
team1 = players_pitch[players_pitch['team_id'] == 1]

territory = {
    0: round((team0['pitch_x_m'] > 52.5).mean() * 100, 1),
    1: round((team1['pitch_x_m'] < 52.5).mean() * 100, 1)
}

print('=== TERRITORY (% time in attacking half) ===')
print(f'Team 0: {territory[0]}%')
print(f'Team 1: {territory[1]}%')

=== TERRITORY (% time in attacking half) ===
Team 0: 51.1%
Team 1: 69.0%


5-Momentum

In [23]:
total_frames  = movement_df['frame'].max()
recent_cutoff = int(total_frames * 0.8)

# movement_df already has team_id — no merge needed
recent_movement = movement_df[
    (movement_df['frame'] >= recent_cutoff) &
    (movement_df['team_id'] >= 0)
].copy()

print(f'Recent frames rows: {len(recent_movement)}')
print(f'Team counts: {recent_movement["team_id"].value_counts().to_dict()}')

recent_speed = recent_movement.groupby('team_id')['speed'].mean()
total_recent = recent_speed.sum()

momentum = {
    0: round(recent_speed.get(0, 0) / total_recent * 100, 1) if total_recent > 0 else 50,
    1: round(recent_speed.get(1, 0) / total_recent * 100, 1) if total_recent > 0 else 50
}

print('=== MOMENTUM (recent activity %) ===')
print(f'Team 0: {momentum[0]}%')
print(f'Team 1: {momentum[1]}%')

Recent frames rows: 3009
Team counts: {1: 1539, 0: 1470}
=== MOMENTUM (recent activity %) ===
Team 0: 56.6%
Team 1: 43.4%


6-Match Outcome Probability

In [24]:
def calculate_match_outcome(possession, shots, territory, momentum, avg_danger):
    """
    Combine all statistics into win/draw/loss probabilities.

    Each stat contributes a score for each team.
    Higher score = more likely to win.
    """

    # Normalize each stat to 0-1 scale
    total_shots    = shots[0] + shots[1] + 1e-6
    total_danger   = avg_danger[0] + avg_danger[1] + 1e-6

    poss_score  = {0: possession[0] / 100,    1: possession[1] / 100}
    shot_score  = {0: shots[0] / total_shots,  1: shots[1] / total_shots}
    terr_score  = {0: territory[0] / 100,      1: territory[1] / 100}
    mom_score   = {0: momentum[0] / 100,       1: momentum[1] / 100}
    danger_score= {0: avg_danger[0] / total_danger, 1: avg_danger[1] / total_danger}

    # Weighted team strength score
    # Shots and danger matter most for predicting outcome
    strength = {}
    for team in [0, 1]:
        strength[team] = (
            0.25 * shot_score[team]   +
            0.25 * danger_score[team] +
            0.20 * poss_score[team]   +
            0.15 * terr_score[team]   +
            0.15 * mom_score[team]
        )

    # Convert strength difference to win probability
    # Using a sigmoid-like function
    diff = strength[0] - strength[1]

    # Base probabilities
    win0  = 1 / (1 + np.exp(-10 * diff))  # sigmoid
    win1  = 1 - win0

    # Draw probability — higher when teams are close
    draw  = max(0, 0.3 - abs(diff) * 2)

    # Normalize so all three sum to 1
    total = win0 + win1 + draw
    win0  = round(win0 / total * 100, 1)
    win1  = round(win1 / total * 100, 1)
    draw  = round(draw / total * 100, 1)

    return win0, draw, win1, strength

win0, draw, win1, strength = calculate_match_outcome(
    possession, shots, territory, momentum, avg_danger
)

print('=== MATCH STATISTICS SUMMARY ===')
print(f'{"Stat":<20} {"Team 0":>10} {"Team 1":>10}')
print(f'{"-"*40}')
print(f'{"Possession":<20} {possession[0]:>9}% {possession[1]:>9}%')
print(f'{"Shots":<20} {shots[0]:>10} {shots[1]:>10}')
print(f'{"Avg Danger":<20} {avg_danger[0]:>10} {avg_danger[1]:>10}')
print(f'{"Territory":<20} {territory[0]:>9}% {territory[1]:>9}%')
print(f'{"Momentum":<20} {momentum[0]:>9}% {momentum[1]:>9}%')
print(f'{"Strength Score":<20} {strength[0]:>10.3f} {strength[1]:>10.3f}')
print()
print('=== MATCH OUTCOME PREDICTION ===')
print(f'Team 0 Win:  {win0}%')
print(f'Draw:        {draw}%')
print(f'Team 1 Win:  {win1}%')

=== MATCH STATISTICS SUMMARY ===
Stat                     Team 0     Team 1
----------------------------------------
Possession                83.4%      16.6%
Shots                         7          2
Avg Danger                0.262      0.266
Territory                 51.1%      69.0%
Momentum                  56.6%      43.4%
Strength Score            0.647      0.383

=== MATCH OUTCOME PREDICTION ===
Team 0 Win:  93.3%
Draw:        0.0%
Team 1 Win:  6.7%


7-Save

In [25]:
# Save match stats and outcome
match_outcome = {
    'possession_team0':  possession[0],
    'possession_team1':  possession[1],
    'shots_team0':       shots[0],
    'shots_team1':       shots[1],
    'avg_danger_team0':  avg_danger[0],
    'avg_danger_team1':  avg_danger[1],
    'territory_team0':   territory[0],
    'territory_team1':   territory[1],
    'momentum_team0':    momentum[0],
    'momentum_team1':    momentum[1],
    'win_prob_team0':    win0,
    'draw_prob':         draw,
    'win_prob_team1':    win1
}

outcome_df = pd.DataFrame([match_outcome])
outcome_df.to_csv('match_predictions.csv', index=False)

print('Saved to match_predictions.csv')
print()
print(outcome_df.T)

Saved to match_predictions.csv

                       0
possession_team0  83.400
possession_team1  16.600
shots_team0        7.000
shots_team1        2.000
avg_danger_team0   0.262
avg_danger_team1   0.266
territory_team0   51.100
territory_team1   69.000
momentum_team0    56.600
momentum_team1    43.400
win_prob_team0    93.300
draw_prob          0.000
win_prob_team1     6.700
